Import libraries

In [ ]:
import pandas as pd
import numpy as np
import os
import gc
import warnings

warnings.filterwarnings("ignore")

print("Libraries loaded successfully.")

Libraries loaded successfully.


Define paths

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/GC210_Retail_Forecasting"
RAW_DIR = f"{PROJECT_DIR}/data/raw"
PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)

Mounted at /content/drive
Project: /content/drive/MyDrive/GC210_Retail_Forecasting
Raw data: /content/drive/MyDrive/GC210_Retail_Forecasting/data/raw
Processed data: /content/drive/MyDrive/GC210_Retail_Forecasting/data/processed


Understand the sales data

Load sales data

In [ ]:
sales_path = f"/content/drive/MyDrive/DATASET/sales_train_validation.csv"

sales = pd.read_csv(sales_path)

print("Sales shape:", sales.shape)

Sales shape: (30490, 1919)


Cell 4 — Examine columns

In [ ]:
print("First 15 columns:")
print(sales.columns[:15].tolist())

print("\nLast 10 columns:")
print(sales.columns[-10:].tolist())

First 15 columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2', 'd_3', 'd_4', 'd_5', 'd_6', 'd_7', 'd_8', 'd_9']

Last 10 columns:
['d_1904', 'd_1905', 'd_1906', 'd_1907', 'd_1908', 'd_1909', 'd_1910', 'd_1911', 'd_1912', 'd_1913']


Separate metadata and daily sales columns

In [ ]:
id_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]

sales_columns = [
    col for col in sales.columns
    if col.startswith("d_")
]

print("Metadata columns:", len(id_columns))
print("Daily sales columns:", len(sales_columns))

print("\nFirst sales columns:")
print(sales_columns[:10])

print("\nLast sales columns:")
print(sales_columns[-10:])

Metadata columns: 6
Daily sales columns: 1913

First sales columns:
['d_1', 'd_2', 'd_3', 'd_4', 'd_5', 'd_6', 'd_7', 'd_8', 'd_9', 'd_10']

Last sales columns:
['d_1904', 'd_1905', 'd_1906', 'd_1907', 'd_1908', 'd_1909', 'd_1910', 'd_1911', 'd_1912', 'd_1913']


Understand products and stores

In [ ]:
print("Unique items :", sales["item_id"].nunique())
print("Unique stores:", sales["store_id"].nunique())
print("Departments  :", sales["dept_id"].nunique())
print("Categories   :", sales["cat_id"].nunique())
print("States       :", sales["state_id"].nunique())

Unique items : 3049
Unique stores: 10
Departments  : 7
Categories   : 3
States       : 3


Category distribution

In [ ]:
category_counts = sales["cat_id"].value_counts()

print(category_counts)

cat_id
FOODS        14370
HOUSEHOLD    10470
HOBBIES       5650
Name: count, dtype: int64


Store distribution

In [ ]:
store_counts = sales["store_id"].value_counts()

print(store_counts)

store_id
CA_1    3049
CA_2    3049
CA_3    3049
CA_4    3049
TX_1    3049
TX_2    3049
TX_3    3049
WI_1    3049
WI_2    3049
WI_3    3049
Name: count, dtype: int64


Understand sales values

Basic sales statistics

In [ ]:
sales_values = sales[sales_columns]

print("Total sales:", sales_values.sum().sum())
print("Mean sales:", sales_values.mean().mean())
print("Maximum daily sales:", sales_values.max().max())
print("Minimum sales:", sales_values.min().min())

Total sales: 65695409
Mean sales: 1.126322153733316
Maximum daily sales: 763
Minimum sales: 0


Check missing values

In [ ]:
print("Missing values in sales:")
print(sales[sales_columns].isna().sum().sum())

Missing values in sales:
0


In [ ]:
print("Rows containing missing sales:")
print(sales[sales_columns].isna().any(axis=1).sum())

Rows containing missing sales:
0


Check zero-demand observations

In [ ]:
zero_count = (sales_values == 0).sum().sum()
total_count = sales_values.size

zero_percentage = zero_count / total_count * 100

print(f"Zero-sales observations: {zero_count:,}")
print(f"Total observations     : {total_count:,}")
print(f"Zero percentage        : {zero_percentage:.2f}%")

Zero-sales observations: 39,777,094
Total observations     : 58,327,370
Zero percentage        : 68.20%


Calendar preprocessing

Load calendar

In [ ]:
calendar_path = f"/content/drive/MyDrive/DATASET/calendar.csv"

calendar = pd.read_csv(calendar_path)

print("Calendar shape:", calendar.shape)

calendar.head()

Calendar shape: (1969, 14)


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


Inspect calendar columns

In [ ]:
print(calendar.columns.tolist())

['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']


Convert date

In [ ]:
calendar["date"] = pd.to_datetime(calendar["date"])

print(calendar["date"].min())
print(calendar["date"].max())

2011-01-29 00:00:00
2016-06-19 00:00:00


Check calendar missing values

In [ ]:
calendar_missing = calendar.isna().sum()

calendar_missing[calendar_missing > 0]

,0
event_name_1,1807
event_type_1,1807
event_name_2,1964
event_type_2,1964


Price preprocessing

Load price data

In [ ]:
prices_path = f"/content/drive/MyDrive/DATASET/sell_prices.csv"

prices = pd.read_csv(prices_path)

print("Prices shape:", prices.shape)

prices.head()

Prices shape: (6841121, 4)


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


Inspect prices

In [ ]:
print(prices.columns.tolist())

['store_id', 'item_id', 'wm_yr_wk', 'sell_price']


Price statistics

In [ ]:
print(prices["sell_price"].describe())

count    6.841121e+06
mean     4.410952e+00
std      3.408814e+00
min      1.000000e-02
25%      2.180000e+00
50%      3.470000e+00
75%      5.840000e+00
max      1.073200e+02
Name: sell_price, dtype: float64


In [ ]:
print("Missing prices:", prices["sell_price"].isna().sum())

Missing prices: 0


Create a development dataset

Select 300 series

In [ ]:
SEED = 42
N_SERIES = 300

series_sample = sales.sample(
    n=N_SERIES,
    random_state=SEED
).copy()

print("Development dataset shape:", series_sample.shape)

Development dataset shape: (300, 1919)


Check selected stores/categories

In [ ]:
print("Selected stores:")
print(series_sample["store_id"].value_counts())

print("\nSelected categories:")
print(series_sample["cat_id"].value_counts())

Selected stores:
store_id
TX_1    50
CA_4    38
WI_2    33
CA_3    30
TX_3    28
WI_3    26
TX_2    26
CA_2    25
WI_1    24
CA_1    20
Name: count, dtype: int64

Selected categories:
cat_id
FOODS        153
HOUSEHOLD     79
HOBBIES       68
Name: count, dtype: int64


Convert sales from wide to long format

Convert sales from wide to long format

In [ ]:
long_sales = series_sample.melt(
    id_vars=id_columns,
    value_vars=sales_columns,
    var_name="d",
    value_name="sales"
)

print("Long-format shape:", long_sales.shape)

Long-format shape: (573900, 8)


Inspect long data

In [ ]:
long_sales.head(10)

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,FOODS_3_180_CA_1_validation,FOODS_3_180,FOODS_3,FOODS,CA_1,CA,d_1,0
1,HOUSEHOLD_2_383_CA_3_validation,HOUSEHOLD_2_383,HOUSEHOLD_2,HOUSEHOLD,CA_3,CA,d_1,2
2,FOODS_3_409_CA_3_validation,FOODS_3_409,FOODS_3,FOODS,CA_3,CA,d_1,0
3,FOODS_1_097_CA_2_validation,FOODS_1_097,FOODS_1,FOODS,CA_2,CA,d_1,0
4,HOBBIES_1_272_TX_2_validation,HOBBIES_1_272,HOBBIES_1,HOBBIES,TX_2,TX,d_1,0
5,HOUSEHOLD_1_064_CA_4_validation,HOUSEHOLD_1_064,HOUSEHOLD_1,HOUSEHOLD,CA_4,CA,d_1,0
6,FOODS_2_368_TX_2_validation,FOODS_2_368,FOODS_2,FOODS,TX_2,TX,d_1,14
7,HOUSEHOLD_1_395_TX_3_validation,HOUSEHOLD_1_395,HOUSEHOLD_1,HOUSEHOLD,TX_3,TX,d_1,5
8,HOUSEHOLD_1_537_CA_1_validation,HOUSEHOLD_1_537,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA,d_1,3
9,HOUSEHOLD_2_410_TX_2_validation,HOUSEHOLD_2_410,HOUSEHOLD_2,HOUSEHOLD,TX_2,TX,d_1,1


Memory usage

In [ ]:
memory_mb = long_sales.memory_usage(deep=True).sum() / (1024 ** 2)

print(f"Memory usage: {memory_mb:.2f} MB")

Memory usage: 229.02 MB


Calendar day mapping

In [ ]:
calendar_mapping = calendar[
    ["d", "date", "wm_yr_wk"]
].copy()

calendar_mapping.head()

,d,date,wm_yr_wk
0,d_1,2011-01-29,11101
1,d_2,2011-01-30,11101
2,d_3,2011-01-31,11101
3,d_4,2011-02-01,11101
4,d_5,2011-02-02,11101


Merge calendar

In [ ]:
long_sales = long_sales.merge(
    calendar_mapping,
    on="d",
    how="left"
)

print(long_sales.shape)

(573900, 10)


Check dates

In [ ]:
long_sales[[
    "id",
    "d",
    "date",
    "sales"
]].head(20)

,id,d,date,sales
0,FOODS_3_180_CA_1_validation,d_1,2011-01-29,0
1,HOUSEHOLD_2_383_CA_3_validation,d_1,2011-01-29,2
2,FOODS_3_409_CA_3_validation,d_1,2011-01-29,0
3,FOODS_1_097_CA_2_validation,d_1,2011-01-29,0
4,HOBBIES_1_272_TX_2_validation,d_1,2011-01-29,0
5,HOUSEHOLD_1_064_CA_4_validation,d_1,2011-01-29,0
6,FOODS_2_368_TX_2_validation,d_1,2011-01-29,14
7,HOUSEHOLD_1_395_TX_3_validation,d_1,2011-01-29,5
8,HOUSEHOLD_1_537_CA_1_validation,d_1,2011-01-29,3
9,HOUSEHOLD_2_410_TX_2_validation,d_1,2011-01-29,1


Create time features

In [ ]:
long_sales["day_of_week"] = long_sales["date"].dt.dayofweek
long_sales["day_of_month"] = long_sales["date"].dt.day
long_sales["week_of_year"] = long_sales["date"].dt.isocalendar().week.astype(int)
long_sales["month"] = long_sales["date"].dt.month
long_sales["year"] = long_sales["date"].dt.year
long_sales["is_weekend"] = (
    long_sales["day_of_week"] >= 5
).astype(int)

Inspect features

In [ ]:
long_sales[[
    "date",
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "year",
    "is_weekend"
]].head(10)

,date,day_of_week,day_of_month,week_of_year,month,year,is_weekend
0,2011-01-29,5,29,4,1,2011,1
1,2011-01-29,5,29,4,1,2011,1
2,2011-01-29,5,29,4,1,2011,1
3,2011-01-29,5,29,4,1,2011,1
4,2011-01-29,5,29,4,1,2011,1
5,2011-01-29,5,29,4,1,2011,1
6,2011-01-29,5,29,4,1,2011,1
7,2011-01-29,5,29,4,1,2011,1
8,2011-01-29,5,29,4,1,2011,1
9,2011-01-29,5,29,4,1,2011,1


Add event information

Merge event information

In [ ]:
event_columns = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

event_data = calendar[
    ["d"] + event_columns
].copy()

long_sales = long_sales.merge(
    event_data,
    on="d",
    how="left"
)

print(long_sales.shape)

(573900, 23)


Handle categorical event values

In [ ]:
categorical_event_columns = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for col in categorical_event_columns:
    long_sales[col] = long_sales[col].fillna("None")

In [ ]:
long_sales[categorical_event_columns].head()

,event_name_1,event_type_1,event_name_2,event_type_2
0,None,None,None,None
1,None,None,None,None
2,None,None,None,None
3,None,None,None,None
4,None,None,None,None


Add selling price

Merge price

In [ ]:
price_columns = [
    "store_id",
    "item_id",
    "wm_yr_wk",
    "sell_price"
]

price_data = prices[price_columns].copy()

long_sales = long_sales.merge(
    price_data,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

print("Final development shape:", long_sales.shape)

Final development shape: (573900, 24)


Check price availability

In [ ]:
print(
    "Missing sell prices:",
    long_sales["sell_price"].isna().sum()
)

print(
    "Missing percentage:",
    long_sales["sell_price"].isna().mean() * 100
)

Missing sell prices: 138404
Missing percentage: 24.116396584770865


Sort the dataset

In [ ]:
long_sales = long_sales.sort_values(
    ["id", "date"]
).reset_index(drop=True)

In [ ]:
long_sales.head(20)

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,year,is_weekend,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_005_TX_3_validation,FOODS_1_005,FOODS_1,FOODS,TX_3,TX,d_1,2,2011-01-29,11101,...,2011,1,None,None,None,None,0,0,0,2.94
1,FOODS_1_005_TX_3_validation,FOODS_1_005,FOODS_1,FOODS,TX_3,TX,d_2,3,2011-01-30,11101,...,2011,1,None,None,None,None,0,0,0,2.94
2,FOODS_1_005_TX_3_validation,FOODS_1_005,FOODS_1,FOODS,TX_3,TX,d_3,3,2011-01-31,11101,...,2011,0,None,None,None,None,0,0,0,2.94
3,FOODS_1_005_TX_3_validation,FOODS_1_005,FOODS_1,FOODS,TX_3,TX,d_4,2,2011-02-01,11101,...,2011,0,None,None,None,None,1,1,0,2.94
4,FOODS_1_005_TX_3_validation,FOODS_1_005,FOODS_1,FOODS,TX_3,TX,d_5,0,2011-02-02,11101,...,2011,0,None,None,None,None,1,0,1,2.94
5,FOODS_1_005_TX_3_validation,FOODS_1_005,FOODS_1,FOODS,TX_3,TX,d_6,2,2011-02-03,11101,...,2011,0,None,None,None,None,1,1,1,2.94
6,FOODS_1_005_TX_3_validation,FOODS_1_005,FOODS_1,FOODS,TX_3,TX,d_7,2,2011-02-04,11101,...,2011,0,None,None,None,None,1,0,0,2.94
7,FOODS_1_005_TX_3_validation,FOODS_1_005,FOODS_1,FOODS,TX_3,TX,d_8,3,2011-02-05,11102,...,2011,1,None,None,None,None,1,1,1,2.94
8,FOODS_1_005_TX_3_validation,FOODS_1_005,FOODS_1,FOODS,TX_3,TX,d_9,2,2011-02-06,11102,...,2011,1,SuperBowl,Sporting,None,None,1,1,1,2.94
9,FOODS_1_005_TX_3_validation,FOODS_1_005,FOODS_1,FOODS,TX_3,TX,d_10,2,2011-02-07,11102,...,2011,0,None,None,None,None,1,1,0,2.94


Check duplicate observations

In [ ]:
duplicates = long_sales.duplicated(
    subset=["id", "date"]
).sum()

print("Duplicate observations:", duplicates)

Duplicate observations: 0


Check date continuity

In [ ]:
example_id = long_sales["id"].iloc[0]

example_series = long_sales[
    long_sales["id"] == example_id
].sort_values("date")

print("Series:", example_id)
print("Number of observations:", len(example_series))
print("Start:", example_series["date"].min())
print("End:", example_series["date"].max())

Series: FOODS_1_005_TX_3_validation
Number of observations: 1913
Start: 2011-01-29 00:00:00
End: 2016-04-24 00:00:00


Basic sales summary by category

In [ ]:
category_summary = (
    long_sales
    .groupby("cat_id")["sales"]
    .agg(
        ["count", "mean", "std", "min", "max", "sum"]
    )
)

category_summary

,count,mean,std,min,max,sum
cat_id,,,,,,
FOODS,292689,1.097062,3.052268,0,634,321098
HOBBIES,130084,0.541881,1.751562,0,67,70490
HOUSEHOLD,151127,0.722710,2.500104,0,72,109221


Basic sales summary by store

In [ ]:
store_summary = (
    long_sales
    .groupby("store_id")["sales"]
    .agg(
        ["count", "mean", "std", "min", "max", "sum"]
    )
)

store_summary

,count,mean,std,min,max,sum
store_id,,,,,,
CA_1,38260,0.869524,1.670909,0,36,33268
CA_2,47825,1.471218,4.659380,0,73,70361
CA_3,57390,1.447778,4.031582,0,72,83088
CA_4,72694,0.585592,1.697304,0,49,42569
TX_1,95650,0.615400,2.547352,0,634,58863
TX_2,49738,0.528871,1.434028,0,54,26305
TX_3,53564,1.435330,3.411935,0,63,76882
WI_1,45912,0.636391,1.268954,0,37,29218
WI_2,63129,0.711923,1.604712,0,28,44943


Save the development dataset

In [ ]:
development_path = (
    f"{PROCESSED_DIR}/m5_development_300_series.parquet"
)

long_sales.to_parquet(
    development_path,
    index=False
)

print("Saved:")
print(development_path)

Saved:
/content/drive/MyDrive/GC210_Retail_Forecasting/data/processed/m5_development_300_series.parquet


Test loading the saved dataset

In [ ]:
test_data = pd.read_parquet(development_path)

print("Loaded successfully.")
print("Shape:", test_data.shape)
print("Columns:", test_data.columns.tolist())

Loaded successfully.
Shape: (573900, 24)
Columns: ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'day_of_week', 'day_of_month', 'week_of_year', 'month', 'year', 'is_weekend', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price']


Clean memory

In [ ]:
del test_data
del series_sample
del long_sales

gc.collect()

print("Memory cleanup completed.")

Memory cleanup completed.
